In [ ]:
# Installing pymongo

!pip install pymongo

In [ ]:
# Importing libraries

from pymongo import MongoClient
import pandas as pd
import numpy as np

In [ ]:
# Convert pandas and NumPy values into normal Python values

def convert(value):
    if isinstance(value, dict):
        return {k: convert(v) for k, v in value.items()}
    elif isinstance(value, list):
        return [convert(v) for v in value]
    elif isinstance(value, np.integer):
        return int(value)
    elif isinstance(value, np.floating):
        return float(value)
    elif isinstance(value, np.bool_):
        return bool(value)
    elif isinstance(value, np.ndarray):
        return value.tolist()
    elif isinstance(value, pd.Timestamp):
        return None if pd.isna(value) else value.to_pydatetime()
    elif pd.isna(value):
        return None
    else:
        return value

In [ ]:
# Connecting to MongoDB Atlas

client = MongoClient("YOUR_MONGODB_ATLAS_CONNECTION_STRING")

# Connecting to Database

db = client["DB"]

# Checking connection

print(db.list_collection_names())

In [ ]:
# Loading datasets

raw_base = "https://raw.githubusercontent.com/THEX3MN/NorthStar-Dataset/main/NorthStar%20Dataset%20(Cleaned)"

customers = pd.read_csv(f"{raw_base}/customers_clean.csv")
orders = pd.read_csv(f"{raw_base}/orders_clean.csv")
complaints = pd.read_csv(f"{raw_base}/complaints_clean.csv")
app_events = pd.read_csv(f"{raw_base}/app_events_clean.csv")

In [ ]:
# Replacing missing values with none

app_events = app_events.where(pd.notnull(app_events), None)
complaints = complaints.where(pd.notnull(complaints), None)
orders = orders.where(pd.notnull(orders), None)
customers = customers.where(pd.notnull(customers), None)

In [ ]:
# Coverting each dataframe into dictionaries

app_events_records = app_events.to_dict("records")
complaints_records = complaints.to_dict("records")
orders_records = orders.to_dict("records")
customers_records = customers.to_dict("records")

In [ ]:
# Selecting the collections

app_events_collection = db["app_events"]
complaints_collection = db["complaints"]
orders_collection = db["orders"]
customers_collection = db["customers"]
service_cases_collection = db["service_cases"]

# Dropping the collections first to avoid duplicate inserts when rerunning the notebook

db.drop_collection("app_events")
db.drop_collection("complaints")
db.drop_collection("orders")
db.drop_collection("customers")
db.drop_collection("service_cases")

# Creating the collections

app_events_collection = db["app_events"]
complaints_collection = db["complaints"]
orders_collection = db["orders"]
customers_collection = db["customers"]
service_cases_collection = db["service_cases"]

# Inserting documents

app_events_result = app_events_collection.insert_many(app_events_records)
complaints_result = complaints_collection.insert_many(complaints_records)
orders_result = orders_collection.insert_many(orders_records)
customers_result = customers_collection.insert_many(customers_records)

# Checking the count of the inseted documents

print("Inserted app_events:", len(app_events_result.inserted_ids))
print("Inserted complaints:", len(complaints_result.inserted_ids))
print("Inserted orders:", len(orders_result.inserted_ids))
print("Inserted customers:", len(customers_result.inserted_ids))

# Display one sample document from each collection

print(app_events_collection.find_one())
print(complaints_collection.find_one())
print(orders_collection.find_one())
print(customers_collection.find_one())

In [ ]:
# Create Operations

# Creating one service case document from the first complaint record

first_case_row = complaints.iloc[0]

first_case_order = orders[orders["order_id"] == first_case_row["order_id"]]
first_case_customer = customers[customers["customer_id"] == first_case_row["customer_id"]]
first_case_events = app_events[
    (app_events["customer_id"] == first_case_row["customer_id"]) &
    (app_events["order_id"] == first_case_row["order_id"])
].head(5)

sample_service_case = {
    "case_id": "CASE_" + str(first_case_row["complaint_id"]),
    "complaint_id": first_case_row["complaint_id"],
    "customer_id": first_case_row["customer_id"],
    "order_id": first_case_row["order_id"],
    "case_type": first_case_row["complaint_type"],
    "priority": first_case_row["severity"],
    "current_status": first_case_row["status"],
    "created_at": first_case_row["created_at"],
    "complaint_case": {
        "channel": first_case_row["channel"],
        "resolution_days": first_case_row["resolution_days"],
        "compensation_amount": first_case_row["compensation_amount"]
    },
    "customer_profile": {
        "age": first_case_customer.iloc[0]["age"] if not first_case_customer.empty else None,
        "home_zone": first_case_customer.iloc[0]["home_zone"] if not first_case_customer.empty else None,
        "customer_type": first_case_customer.iloc[0]["customer_type"] if not first_case_customer.empty else None,
        "loyalty_score": first_case_customer.iloc[0]["loyalty_score"] if not first_case_customer.empty else None,
        "app_engagement_score": first_case_customer.iloc[0]["app_engagement_score"] if not first_case_customer.empty else None,
        "preferred_channel": first_case_customer.iloc[0]["preferred_channel"] if not first_case_customer.empty else None,
        "account_status": first_case_customer.iloc[0]["account_status"] if not first_case_customer.empty else None
    },
    "order_context": {
        "service_type": first_case_order.iloc[0]["service_type"] if not first_case_order.empty else None,
        "order_created_at": first_case_order.iloc[0]["order_created_at"] if not first_case_order.empty else None,
        "promised_window_hours": first_case_order.iloc[0]["promised_window_hours"] if not first_case_order.empty else None,
        "pickup_zone": first_case_order.iloc[0]["pickup_zone"] if not first_case_order.empty else None,
        "dropoff_zone": first_case_order.iloc[0]["dropoff_zone"] if not first_case_order.empty else None,
        "priority_level": first_case_order.iloc[0]["priority_level"] if not first_case_order.empty else None,
        "order_value": first_case_order.iloc[0]["order_value"] if not first_case_order.empty else None,
        "special_handling_flag": first_case_order.iloc[0]["special_handling_flag"] if not first_case_order.empty else None
    },
    "event_history": first_case_events[
        ["event_id", "event_timestamp", "event_type", "session_id", "device_type", "zone_context", "api_latency_ms", "success_flag"]
    ].to_dict("records")
}

# Converting the service case document into MongoDB-safe values

sample_service_case = convert(sample_service_case)

# Inserting one service case document

service_cases_collection.insert_one(sample_service_case)

# Creating many service case documents from the next complaint records

service_case_docs = []

for _, row in complaints.iloc[1:6].iterrows():
    linked_order = orders[orders["order_id"] == row["order_id"]]
    linked_customer = customers[customers["customer_id"] == row["customer_id"]]
    linked_events = app_events[
        (app_events["customer_id"] == row["customer_id"]) &
        (app_events["order_id"] == row["order_id"])
    ].head(5)

    case_doc = {
        "case_id": "CASE_" + str(row["complaint_id"]),
        "complaint_id": row["complaint_id"],
        "customer_id": row["customer_id"],
        "order_id": row["order_id"],
        "case_type": row["complaint_type"],
        "priority": row["severity"],
        "current_status": row["status"],
        "created_at": row["created_at"],
        "complaint_case": {
            "channel": row["channel"],
            "resolution_days": row["resolution_days"],
            "compensation_amount": row["compensation_amount"]
        },
        "customer_profile": {
            "age": linked_customer.iloc[0]["age"] if not linked_customer.empty else None,
            "home_zone": linked_customer.iloc[0]["home_zone"] if not linked_customer.empty else None,
            "customer_type": linked_customer.iloc[0]["customer_type"] if not linked_customer.empty else None,
            "loyalty_score": linked_customer.iloc[0]["loyalty_score"] if not linked_customer.empty else None,
            "app_engagement_score": linked_customer.iloc[0]["app_engagement_score"] if not linked_customer.empty else None,
            "preferred_channel": linked_customer.iloc[0]["preferred_channel"] if not linked_customer.empty else None,
            "account_status": linked_customer.iloc[0]["account_status"] if not linked_customer.empty else None
        },
        "order_context": {
            "service_type": linked_order.iloc[0]["service_type"] if not linked_order.empty else None,
            "order_created_at": linked_order.iloc[0]["order_created_at"] if not linked_order.empty else None,
            "promised_window_hours": linked_order.iloc[0]["promised_window_hours"] if not linked_order.empty else None,
            "pickup_zone": linked_order.iloc[0]["pickup_zone"] if not linked_order.empty else None,
            "dropoff_zone": linked_order.iloc[0]["dropoff_zone"] if not linked_order.empty else None,
            "priority_level": linked_order.iloc[0]["priority_level"] if not linked_order.empty else None,
            "order_value": linked_order.iloc[0]["order_value"] if not linked_order.empty else None,
            "special_handling_flag": linked_order.iloc[0]["special_handling_flag"] if not linked_order.empty else None
        },
        "event_history": linked_events[
            ["event_id", "event_timestamp", "event_type", "session_id", "device_type", "zone_context", "api_latency_ms", "success_flag"]
        ].to_dict("records")
    }

    service_case_docs.append(case_doc)

# Converting many service case documents into MongoDB-safe values

service_case_docs = [convert(doc) for doc in service_case_docs]

# Inserting many service case documents

if len(service_case_docs) > 0:
    service_cases_result = service_cases_collection.insert_many(service_case_docs)
    print("Inserted service case documents:", len(service_cases_result.inserted_ids))

# Checking the number of documents in the new collection

print("Service cases count:", service_cases_collection.count_documents({}))

# Viewing one sample document from the new collection

print(service_cases_collection.find_one())

In [ ]:
# Creating indexes for the service_cases collection

service_cases_collection.create_index("case_id")
service_cases_collection.create_index("customer_id")
service_cases_collection.create_index("order_id")
service_cases_collection.create_index("priority")
service_cases_collection.create_index("current_status")

# Confirming the indexes that were created for service_cases

print("Service Cases Indexes:")
print(service_cases_collection.index_information())

In [ ]:
# Read Operations

# Viewing all collections in the database

print(db.list_collection_names())

# Finding customers from a specific customer type

sample_customer_type = "Consumer" if "Consumer" in customers["customer_type"].dropna().unique() else customers.iloc[0]["customer_type"]
consumer_customers = list(customers_collection.find({"customer_type": sample_customer_type}))
print("Number of customers in type", sample_customer_type, ":", len(consumer_customers))
print(consumer_customers[:2])

# Finding orders from a specific service type

sample_service_type = "Parcel" if "Parcel" in orders["service_type"].dropna().unique() else orders.iloc[0]["service_type"]
parcel_orders = list(orders_collection.find({"service_type": sample_service_type}))
print("Number of orders in service type", sample_service_type, ":", len(parcel_orders))
print(parcel_orders[:2])

# Finding complaints with high severity

high_severity_complaints = list(complaints_collection.find({"severity": "High"}))
print("Number of high severity complaints:", len(high_severity_complaints))
print(high_severity_complaints[:2])

# Finding app events from a specific event type

sample_event_type = "track_order" if "track_order" in app_events["event_type"].dropna().unique() else app_events.iloc[0]["event_type"]
track_events = list(app_events_collection.find({"event_type": sample_event_type}))
print("Number of", sample_event_type, "events:", len(track_events))
print(track_events[:2])

# Finding all complaints for one customer

sample_customer_id = customers.iloc[0]["customer_id"]

customer_complaints = list(complaints_collection.find({"customer_id": sample_customer_id}))
print("Complaints for customer", sample_customer_id, ":", len(customer_complaints))
print(customer_complaints[:2])

# Finding all orders for one customer

customer_orders = list(orders_collection.find({"customer_id": sample_customer_id}))
print("Orders for customer", sample_customer_id, ":", len(customer_orders))
print(customer_orders[:2])

# Finding app events for one customer

customer_events = list(app_events_collection.find({"customer_id": sample_customer_id}))
print("App events for customer", sample_customer_id, ":", len(customer_events))
print(customer_events[:2])

# Finding unresolved complaints

unresolved_complaints = list(complaints_collection.find({"status": {"$ne": "Resolved"}}))
print("Unresolved complaints:", len(unresolved_complaints))
print(unresolved_complaints[:2])

# Finding orders with priority level High

sample_priority_level = "High" if "High" in orders["priority_level"].dropna().unique() else orders.iloc[0]["priority_level"]
high_priority_orders = list(orders_collection.find({"priority_level": sample_priority_level}))
print("Orders with priority level", sample_priority_level, ":", len(high_priority_orders))
print(high_priority_orders[:2])

# Finding app events with failed success flag

failed_events = list(app_events_collection.find({"success_flag": 0}))
print("Failed app events:", len(failed_events))
print(failed_events[:2])

In [ ]:
# Update Operations
# Updating one service case

sample_case = service_cases_collection.find_one()

if sample_case is not None:
    service_cases_collection.update_one(
        {"case_id": sample_case["case_id"]},
        {"$set": {
            "current_status": "Resolved",
            "resolution_notes": "Issue resolved after customer follow-up",
            "resolved_by": "Operations Team"
        }}
    )
    print("Service case updated successfully.")
    print(service_cases_collection.find_one({"case_id": sample_case["case_id"]}))
else:
    print("No service case record found to update.")

# Updating many open, high-severity complaints

complaints_collection.update_many(
    {"severity": "High", "status": "Open"},
    {"$set": {
        "escalation_required": 1,
        "priority_review": "Yes"
    }}
)

print("High severity open complaints updated.")

# Updating many app events with failed success_flag values

app_events_collection.update_many(
    {"success_flag": 0},
    {"$set": {
        "retry_required": 1,
        "operational_status": "Failed"
    }}
)

print("Failed app events updated.")

# Replacing one customer document

sample_customer = customers_collection.find_one()

if sample_customer is not None:
    replacement_customer = {k: v for k, v in sample_customer.items() if k != "_id"}

    replacement_customer["preferred_channel"] = "App"
    replacement_customer["account_status"] = "Active"

    customers_collection.replace_one(
        {"customer_id": replacement_customer["customer_id"]},
        replacement_customer
    )

    print("Customer document replaced successfully.")
    print(customers_collection.find_one({"customer_id": replacement_customer["customer_id"]}))
else:
    print("No customer record found to replace.")

In [ ]:
# Delete Operations

# Creating a small archive collection for safe delete demonstrations

db.drop_collection("complaints_archive")
complaints_archive_collection = db["complaints_archive"]

# Inserting a small sample copy of complaint documents into the archive collection

archive_records = [convert(doc) for doc in complaints.head(15).to_dict("records")]
complaints_archive_collection.insert_many(archive_records)

# Checking the number of documents before deletion

print("Complaints archive count before delete:", complaints_archive_collection.count_documents({}))

# Deleting one complaint document from the archive collection

first_archive_doc = complaints_archive_collection.find_one()

if first_archive_doc is not None:
    deleted_one_result = complaints_archive_collection.delete_one(
        {"complaint_id": first_archive_doc["complaint_id"]}
    )
    print("One complaint document deleted from the archive collection.")
    print("Deleted documents:", deleted_one_result.deleted_count)

# Deleting many complaint documents from the archive collection

archive_status = first_archive_doc["status"] if first_archive_doc is not None else None

if archive_status is not None:
    deleted_many_result = complaints_archive_collection.delete_many(
        {"status": archive_status}
    )
    print("Complaint documents with status", archive_status, "deleted from the archive collection.")
    print("Deleted documents:", deleted_many_result.deleted_count)

# Checking the number of documents after deletion

print("Complaints archive count after delete:", complaints_archive_collection.count_documents({}))

# Viewing a remaining sample document from the archive collection

print(complaints_archive_collection.find_one())

In [ ]:
# Sorting Queries

# Sorting complaints by resolution days in descending order

sorted_complaints = list(
    complaints_collection.find().sort("resolution_days", -1).limit(5)
)
print("Top 5 complaints with the longest resolution time:")
print(sorted_complaints)

# Sorting orders by order value in descending order

sorted_orders = list(
    orders_collection.find().sort("order_value", -1).limit(5)
)
print("Top 5 highest value orders:")
print(sorted_orders)

# Sorting app events by API latency in descending order

sorted_events = list(
    app_events_collection.find().sort("api_latency_ms", -1).limit(5)
)
print("Top 5 slowest app events:")
print(sorted_events)

In [ ]:
# Checking Collection Sizes Again

print("Customers documents:", customers_collection.count_documents({}))
print("Orders documents:", orders_collection.count_documents({}))
print("Complaints documents:", complaints_collection.count_documents({}))
print("App events documents:", app_events_collection.count_documents({}))